# Etapa 2 ETL Bronze: PostgreSQL → RustFS (Parquet)

Lee las 9 tablas del esquema `olist` desde PostgreSQL vía JDBC y las escribe
como Parquet en la capa bronze de RustFS (`s3a://olist/bronze/<tabla>/`).

**Prerequisitos:**
- Stack de Etapa 2 levantado: `docker compose up -d --build` desde `etapa2/`.
- Stack de Etapa 1 levantado (PostgreSQL): `docker compose up -d` desde la raíz del proyecto.
- Datos cargados en el esquema `olist` (scripts de Etapa 1 ya ejecutados).

**Capa bronze:** copia fiel de la fuente, sin limpieza. Solo cambia el formato (Parquet)
y el transporte (de RDBMS a object storage).

## 1. SparkSession

In [1]:
import os
from pyspark.sql import SparkSession

PG_HOST  = os.environ.get('POSTGRES_HOST', 'postgres')
PG_PORT  = os.environ.get('POSTGRES_PORT', '5432')
PG_DB    = os.environ.get('POSTGRES_DB',   'olist_db')
PG_USER  = os.environ.get('POSTGRES_USER', 'olist_admin')
PG_PASS  = os.environ.get('POSTGRES_PASSWORD', '')

RUSTFS_ENDPOINT = os.environ.get('RUSTFS_ENDPOINT', 'http://rustfs:9000')
S3_KEY  = os.environ.get('AWS_ACCESS_KEY_ID',     'rustfsadmin')
S3_SEC  = os.environ.get('AWS_SECRET_ACCESS_KEY', 'rustfsadmin123')

JARS = ','.join([
    '/opt/spark/extra-jars/iceberg-spark-runtime-4.1_2.13-1.11.0.jar',
    '/opt/spark/extra-jars/hadoop-aws-3.4.2.jar',
    '/opt/spark/extra-jars/aws-sdk-bundle-v2-2.29.52.jar',
    '/opt/spark/extra-jars/postgresql-42.7.4.jar',
])

spark = (
    SparkSession.builder
    .appName('etapa2-bronze-etl')
    .master('local[*]')
    .config('spark.jars', JARS)
    # S3A → RustFS
    .config('spark.hadoop.fs.s3a.impl',                   'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.endpoint',               RUSTFS_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',             S3_KEY)
    .config('spark.hadoop.fs.s3a.secret.key',             S3_SEC)
    .config('spark.hadoop.fs.s3a.path.style.access',      'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    # Iceberg
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.lake',           'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.lake.type',      'hadoop')
    .config('spark.sql.catalog.lake.warehouse', 's3a://olist/warehouse')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} — master: {spark.sparkContext.master}')
print(f'PostgreSQL: jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}')
print(f'RustFS:     {RUSTFS_ENDPOINT}')

26/10/12 13:00:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.1.3 — master: local[*]
PostgreSQL: jdbc:postgresql://postgres:5432/olist_db
RustFS:     http://rustfs:9000


## 2. Tablas a extraer

Las 9 tablas del esquema `olist`. Para las tablas con muchas filas (`geolocation`)
se usa lectura particionada por JDBC para aprovechar el paralelismo de Spark.

In [2]:
JDBC_URL = f'jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}'
JDBC_PROPS = {
    'user':     PG_USER,
    'password': PG_PASS,
    'driver':   'org.postgresql.Driver',
    'fetchsize': '10000',
}

# Tablas reales del schema olist (verificadas contra PostgreSQL).
# (tabla, columna_particion_o_None, num_partitions)
# geolocation tiene ~1 M filas → lectura particionada.
TABLES = [
    ('orders',               None,             1),
    ('order_items',          None,             1),
    ('order_payments',       None,             1),
    ('order_reviews',        None,             1),
    ('customers',            None,             1),
    ('sellers',              None,             1),
    ('products',             None,             1),
    ('category_translation', None,             1),
    ('geo_cep',              None,             1),
    ('geolocation',          'geolocation_id', 4),
]

BRONZE_BASE = 's3a://olist/bronze'
print(f'Destino bronze: {BRONZE_BASE}/')
print(f'Tablas a extraer: {len(TABLES)}')

Destino bronze: s3a://olist/bronze/
Tablas a extraer: 10


## 3. ETL: PostgreSQL → Bronze

Cada tabla se lee via JDBC y se escribe como Parquet en `s3a://olist/bronze/<tabla>/`.
Modo `overwrite` para que el notebook sea idempotente.

In [3]:
import time

resultados = []

for tabla, col_part, n_part in TABLES:
    t0 = time.time()
    ruta = f'{BRONZE_BASE}/{tabla}'

    if col_part:
        # Leer rango de la columna de partición
        bounds = spark.read.jdbc(
            url=JDBC_URL,
            table=f'(SELECT MIN({col_part}), MAX({col_part}) FROM olist.{tabla}) t',
            properties=JDBC_PROPS,
        ).collect()[0]
        lo, hi = int(bounds[0]), int(bounds[1])
        df = spark.read.jdbc(
            url=JDBC_URL,
            table=f'olist.{tabla}',
            column=col_part,
            lowerBound=lo,
            upperBound=hi,
            numPartitions=n_part,
            properties=JDBC_PROPS,
        )
    else:
        df = spark.read.jdbc(
            url=JDBC_URL,
            table=f'olist.{tabla}',
            properties=JDBC_PROPS,
        )

    n_filas = df.count()
    df.write.mode('overwrite').parquet(ruta)
    elapsed = time.time() - t0

    resultados.append((tabla, n_filas, round(elapsed, 1)))
    print(f'  {tabla:<40} {n_filas:>8} filas  {elapsed:5.1f}s  → {ruta}')

print(f'\nETL completado: {len(resultados)} tablas')

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


  orders                                      99441 filas    6.8s  → s3a://olist/bronze/orders


  order_items                                112650 filas    1.8s  → s3a://olist/bronze/order_items


  order_payments                             103886 filas    0.9s  → s3a://olist/bronze/order_payments


  order_reviews                               99224 filas    1.2s  → s3a://olist/bronze/order_reviews


...

  sellers                                      3095 filas    0.4s  → s3a://olist/bronze/sellers


  products                                    32951 filas    0.5s  → s3a://olist/bronze/products


  category_translation                           71 filas    0.3s  → s3a://olist/bronze/category_translation


  geo_cep                                     19015 filas    0.5s  → s3a://olist/bronze/geo_cep


  geolocation                               1000163 filas    1.8s  → s3a://olist/bronze/geolocation

ETL completado: 10 tablas


## 4. Verificación: leer de vuelta desde bronze

In [4]:
import pandas as pd

filas_bronze = []
for tabla, n_orig, _ in resultados:
    ruta = f'{BRONZE_BASE}/{tabla}'
    df_b = spark.read.parquet(ruta)
    n_b  = df_b.count()
    ok   = '✓' if n_b == n_orig else f'✗ (esperado {n_orig})'
    filas_bronze.append({'tabla': tabla, 'filas_pg': n_orig, 'filas_bronze': n_b, 'ok': ok})

resumen = pd.DataFrame(filas_bronze)
print(resumen.to_string(index=False))

               tabla  filas_pg  filas_bronze ok
              orders     99441         99441  ✓
         order_items    112650        112650  ✓
      order_payments    103886        103886  ✓
       order_reviews     99224         99224  ✓
           customers     99441         99441  ✓
             sellers      3095          3095  ✓
            products     32951         32951  ✓
category_translation        71            71  ✓
             geo_cep     19015         19015  ✓
         geolocation   1000163       1000163  ✓


## 5. Inventario de archivos en el bucket

In [5]:
from py4j.java_gateway import java_import

java_import(spark._jvm, 'org.apache.hadoop.fs.Path')
java_import(spark._jvm, 'org.apache.hadoop.fs.FileSystem')

sc  = spark.sparkContext
jvm = sc._jvm

base_path = jvm.Path(BRONZE_BASE)
fs = base_path.getFileSystem(sc._jsc.hadoopConfiguration())

total_bytes = 0
header = "{:<40} {:>8} {:>10}".format('Tabla', 'Archivos', 'Tamaño')
print(header)
print('-' * 62)
for status in fs.listStatus(base_path):
    tabla_path = status.getPath()
    nombre     = tabla_path.getName()
    archivos   = [f for f in fs.listStatus(tabla_path)
                  if f.getPath().getName().endswith('.parquet')]
    bytes_t    = sum(f.getLen() for f in archivos)
    total_bytes += bytes_t
    print("{:<40} {:>8} {:>9.1f} KB".format(nombre, len(archivos), bytes_t/1024))

print('-' * 62)
print("{:<40} {:>8} {:>9.2f} MB".format('TOTAL', '', total_bytes/1024/1024))

Tabla                                    Archivos     Tamaño
--------------------------------------------------------------
category_translation                            1       3.2 KB
customers                                       1    6828.0 KB
geo_cep                                         1     446.6 KB
geolocation                                     4   20789.0 KB
order_items                                     1    6604.2 KB
order_payments                                  1    3726.0 KB
order_reviews                                   1    9389.2 KB
orders                                          1   10389.9 KB
products                                        1    1331.9 KB
sellers                                         1     128.3 KB
--------------------------------------------------------------
TOTAL                                                 58.24 MB


In [6]:
spark.stop()
print('SparkSession cerrada. Bronze listo para EDA.')

SparkSession cerrada. Bronze listo para EDA.
